In [1]:
import torch
import torchvision
from torchvision import models
import torch.nn as nn

In [ ]:
IMAGENET_PATH = ""

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
convnext_tiny = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
convnext_tiny = convnext_tiny.to(device)

In [ ]:
convnext_tiny.eval()

for param in convnext_tiny.parameters():
    param.requires_grad = False

convnext_tiny

In [ ]:
convnext_tiny_features = convnext_tiny.features
convnext_tiny_features.eval()
convnext_tiny_features

In [6]:
class Codebook(nn.Module):
    def __init__(self, num_entries: int, embedding_dim: int):
        super(Codebook, self).__init__()
        self.embeddings = nn.Embedding(num_entries, embedding_dim)

        nn.init.uniform_(
            self.embeddings.weight, a=-1.0, b=1.0
        )  # Initialize codebook vectors

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # assume the input has a batch dimension
        assert len(x.shape) == 4, "Input tensor must have a batch dimension"

        # Flatten and permute to (batch, H * W, C)
        batch_size, channels, height, width = x.shape
        x_flat = x.view(batch_size, channels, height * width).permute(
            0, 2, 1
        )  # (batch, H * W, C)

        # Normalize input vectors and codebook vectors
        x_normalized = torch.functional.F.normalize(x_flat, dim=-1)  # [B, H * W, C]
        codebook_normalized = torch.functional.F.normalize(
            self.embeddings.weight, dim=-1
        )  # [num_entries, C]

        # Compute cosine similarity between input vectors and codebook vectors
        similarity = torch.matmul(
            x_normalized, codebook_normalized.t()
        )  # [B, H * W, num_entries]

        # Find the closest codebook vector for each spatial token
        indices = torch.argmax(similarity, dim=-1)  # [B, H * W]

        # Replace each token with its closest codebook vector
        quantized = self.embeddings(indices)  # [B, H * W, C]

        # Reshape back to (B, C, H, W)
        quantized = quantized.view(batch_size, height, width, channels).permute(
            0, 3, 1, 2
        )  # [B, C, H, W]

        return quantized

Test for the codebook (check if the codebook output matches the input dimensions)

In [7]:
test_codebook = Codebook(num_entries=1, embedding_dim=768)
test_codebook = test_codebook.to(device)
codebook_vector = (
    test_codebook.embeddings.weight.unsqueeze(-1).unsqueeze(-1).expand(1, 768, 7, 7)
)

t_input = torch.randn(1, 3, 224, 224).to(device)
test_output = convnext_tiny_features(t_input)
test_quantized_vectors = test_codebook(test_output)

assert test_quantized_vectors.shape == test_output.shape

# check if all vectors are close to the codebook vector
assert torch.allclose(test_quantized_vectors, codebook_vector)

Define the codebook and train the codebook with random input data.

In [8]:
codebook = Codebook(num_entries=50, embedding_dim=768)
codebook = codebook.to(device)

In [ ]:
# inject the codebook into the model
convnext_tiny_features.add_module("codebook", codebook)

for module in convnext_tiny_features:
    module.requires_grad_(False)

    if isinstance(module, Codebook):
        module.requires_grad_(True)

convnext_tiny_features

In [ ]:
imagenet_ds = torchvision.datasets.ImageNet(
    root=IMAGENET_PATH, split="val", transform=torchvision.transforms.ToTensor()
)

In [ ]:
BATCH_SIZE = 32
OPTIMIZER_LR = 1e-3
NUM_EPOCHS = 5

In [ ]:
imgnet_val_dl = torch.utils.data.DataLoader(
    imagenet_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4
)

In [ ]:
# write a training loop to train the model

optimizer = torch.optim.Adam(convnext_tiny_features.parameters(), lr=OPTIMIZER_LR)

for epoch in range(NUM_EPOCHS):
    for i, (img, _) in enumerate(imgnet_val_dl):
        img = img.to(device)

        optimizer.zero_grad()
        output = convnext_tiny_features(img)
        loss = torch.norm(output)
        loss.backward()
        optimizer.step()

        if i % 100 == 0:
            print(f"Epoch: {epoch}, Iteration: {i}, Loss: {loss.item()}")